## Code to generate the m-climate for different variables

In [1]:
# imports
import xarray as xr
import numpy as np
import pandas as pd
import glob



In [25]:
input_path = "D:\\Hiwi\\ECMWF_Download\\T_min_max_6h\\*.nc"
output_path = "D:\\Hiwi\\ECMWF_Download\\m-climate_"

for path in glob.glob(input_path):
    data = xr.open_dataset(path)

    # sample data[step] down to weekly values
    data = data.resample(step = "7D").mean()

    # change step into datetime format
    data["step"] = data["time"].values[-1] + data["step"] - pd.Timedelta("6h")

    # stack number and time dimension
    data = data.stack({"numbertime" : ("number","time")})

    # compute quantiles
    quantiles = np.linspace(0,1,101)
    data = data.quantile(quantiles,dim="numbertime",method="linear")

    # rename step dim into time dim
    data = data.rename({"step":"time"})

    data.to_netcdf(f"{output_path}{path[-13:]}")

In [3]:
xr.open_dataset("D:\\Hiwi\\Github_Kenya\\ECMWF-S2S4AFRICA\\m-climate\\m-climate_2025-01-01.nc")

<xarray.Dataset> Size: 16MB
Dimensions:    (time: 6, quantile: 101, latitude: 40, longitude: 42)
Coordinates:
  * time       (time) datetime64[ns] 48B 2025-01-01 2025-01-08 ... 2025-02-05
  * quantile   (quantile) int32 404B 113793886 114907999 ... 190878303
  * latitude   (latitude) float64 320B -36.0 -34.5 -33.0 ... 19.5 21.0 22.5
  * longitude  (longitude) float64 336B -3.0 -1.5 0.0 1.5 ... 55.5 57.0 58.5
Data variables:
    avg_2t     (time, quantile, latitude, longitude) float64 8MB ...
    tp         (time, quantile, latitude, longitude) float64 8MB ...
Attributes:
    CDI:          Climate Data Interface version 2.0.4 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Sun Dec 28 15:33:59 2025: cdo remapcon,targetgrid.txt yipin...
    CDO:          Climate Data Operators version 2.0.4 (https://mpimet.mpg.de...